# 面试题：如何从零实现 PPO 的 Actor-Critic、GAE 与 clipped objective？

## 面试回答主线

PPO 先用旧策略采集 on-policy 轨迹，利用 critic 与 GAE 估计 advantage，再多轮优化新旧动作概率比 $r_t=\exp(\log\pi_	heta-\log\pi_{old})$。actor 最大化 `min(r*A, clip(r,1-eps,1+eps)*A)`，critic 拟合 return，熵项防止策略过早确定。关键合同是：旧 log-prob 必须在采样时冻结、done 后不能 bootstrap、非法动作要在采样和训练时使用同一 mask。下面不用 Gym、Stable-Baselines 或 PPO Trainer，而是手写环境、策略价值网络、轨迹账本、GAE 和更新循环。

## 真实案例：有限人工席位的高风险工单分配

训练班次各有 4 张工单且含两张高风险；测试班次改为 4–6 步并包含一到三张高风险，人工预算只有 2 次。高风险交给人工得 +3，交给机器人得 -3；低风险交给机器人得 +1，浪费人工得 -1。状态包含当前风险、剩余人工预算和剩余步骤。我们用 6 个四步序列训练，再用 6 个未见长度与风险比例的序列测试，共 55 个决策点；这是受控资源分配实验，不代表真实客服策略。

In [1]:
import math  # 导入指数相关数学概念说明所需工具。
import torch  # 导入 PyTorch 以手写策略价值网络和真实反向传播。
from torch import nn  # 导入基础模块与可学习参数。
import torch.nn.functional as F  # 导入底层均方误差工具。
torch.set_num_threads(1)  # 固定小型强化学习实验为单线程。
torch.manual_seed(73)  # 固定策略采样与参数初始化。
train_sequences = [  # 定义六个训练班次的高低风险到达顺序。
    [1, 0, 1, 0],  # 高风险与低风险交替到达。
    [0, 1, 0, 1],  # 从低风险开始交替。
    [1, 1, 0, 0],  # 两张高风险连续先到。
    [0, 0, 1, 1],  # 高风险集中在班次末尾。
    [1, 0, 0, 1],  # 高风险分布在首尾。
    [0, 1, 1, 0],  # 高风险分布在中间。
]  # 完成覆盖全部关键顺序的训练场景。
test_sequences = [  # 定义训练中从未出现的长度与风险比例测试班次。
    [1, 0, 1, 0, 1],  # 五步含三张高风险，人工预算必然不够。
    [0, 1, 0, 0, 1],  # 五步后置两张高风险。
    [1, 0, 1, 0, 0, 0],  # 六步前半段两张高风险。
    [0, 1, 1, 0, 1, 0],  # 六步含三张高风险并从低风险开始。
    [1, 0, 0, 0],  # 四步但只有一张高风险。
    [0, 0, 1, 0, 0],  # 五步且只有一张高风险。
]  # 完成六个未见 horizon 与类别比例的留出评估序列。
def state_vector(risk, remaining_budget, remaining_steps, horizon):  # 把业务状态转换成三维连续向量。
    return torch.tensor([float(risk), remaining_budget / 2.0, remaining_steps / float(horizon)])  # 按当前班次长度归一化剩余时长。
def transition_reward(risk, action):  # 根据风险与路由动作计算即时业务收益。
    if risk == 1 and action == 1:  # 高风险正确交给人工。
        return 3.0  # 返回避免重大损失的正收益。
    if risk == 1 and action == 0:  # 高风险错误交给机器人。
        return -3.0  # 返回自动处理失败的高惩罚。
    if risk == 0 and action == 1:  # 低风险占用稀缺人工。
        return -1.0  # 返回席位浪费成本。
    return 1.0  # 低风险交给机器人得到效率收益。
print("班次  风险顺序       机器人全接回报  最优回报")  # 输出场景输入与可达收益表头。
for index, sequence in enumerate(train_sequences + test_sequences, start=1):  # 遍历十二个业务班次。
    bot_return = sum(transition_reward(risk, 0) for risk in sequence)  # 计算全部机器人处理的基线回报。
    high_count = sum(sequence)  # 统计当前班次高风险工单数。
    human_high_count = min(high_count, 2)  # 最优策略最多把两个人工名额给高风险。
    optimal_return = 3.0 * human_high_count - 3.0 * (high_count - human_high_count) + (len(sequence) - high_count)  # 计算预算不足时仍必须机器人处理多余高风险的真实上界。
    print(f"{index:>2}    {sequence}       {bot_return:>5.1f}          {optimal_return:>5.1f}")  # 展示风险顺序与收益差距。

班次  风险顺序       机器人全接回报  最优回报
 1    [1, 0, 1, 0]        -4.0            8.0
 2    [0, 1, 0, 1]        -4.0            8.0
 3    [1, 1, 0, 0]        -4.0            8.0
 4    [0, 0, 1, 1]        -4.0            8.0
 5    [1, 0, 0, 1]        -4.0            8.0
 6    [0, 1, 1, 0]        -4.0            8.0
 7    [1, 0, 1, 0, 1]        -7.0            5.0
 8    [0, 1, 0, 0, 1]        -3.0            9.0
 9    [1, 0, 1, 0, 0, 0]        -2.0           10.0
10    [0, 1, 1, 0, 1, 0]        -6.0            6.0
11    [1, 0, 0, 0]         0.0            6.0
12    [0, 0, 1, 0, 0]         1.0            7.0


## Baseline（基线）：所有工单都交给机器人

在六个四步训练模板里，这一策略成本最低，却会漏掉两张高风险工单：回报固定为 -4；把两个名额留给高风险时回报为 8。后面的留出序列长度和高风险数量不同，因此测试基线与上界会逐条重算，不沿用 -4/8。

In [2]:
def evaluate_fixed_policy(sequences, choose_human):  # 在多个班次上执行给定确定性规则。
    returns = []  # 收集每个班次的累计回报。
    action_logs = []  # 收集逐步动作以便检查预算。
    for sequence in sequences:  # 遍历所有待评估风险顺序。
        remaining_budget = 2  # 每个新班次重置两个人工名额。
        episode_return = 0.0  # 初始化当前班次累计收益。
        episode_actions = []  # 保存当前班次动作序列。
        for risk in sequence:  # 按到达顺序处理每张工单。
            requested_action = int(choose_human(risk, remaining_budget))  # 调用策略决定是否使用人工。
            action = requested_action if remaining_budget > 0 else 0  # 预算耗尽时强制回退机器人。
            remaining_budget -= int(action == 1)  # 只有实际人工处理才扣减预算。
            episode_return += transition_reward(risk, action)  # 累加当前工单业务收益。
            episode_actions.append(action)  # 记录真实执行动作。
        returns.append(episode_return)  # 保存当前班次总回报。
        action_logs.append(episode_actions)  # 保存当前班次动作账本。
    return returns, action_logs  # 返回逐班次收益和动作。
baseline_returns, baseline_actions = evaluate_fixed_policy(test_sequences, lambda risk, budget: 0)  # 执行全机器人基线。
oracle_returns, oracle_actions = evaluate_fixed_policy(test_sequences, lambda risk, budget: risk == 1)  # 执行知道风险的可解释最优规则。
baseline_mean_return = sum(baseline_returns) / len(baseline_returns)  # 计算基线平均班次回报。
oracle_mean_return = sum(oracle_returns) / len(oracle_returns)  # 计算最优规则平均班次回报。
print("班次  风险顺序      全机器人动作  回报  最优动作      回报")  # 输出逐班次基线对照表头。
for index, (sequence, bot_actions, bot_return, best_actions, best_return) in enumerate(zip(test_sequences, baseline_actions, baseline_returns, oracle_actions, oracle_returns), start=1):  # 对齐场景、动作和收益。
    print(f"{index:>2}    {sequence}    {bot_actions}  {bot_return:>4.1f}  {best_actions}  {best_return:>4.1f}")  # 展示资源分配差异。
print(f"平均班次回报：全机器人={baseline_mean_return:.1f}，可解释上界={oracle_mean_return:.1f}")  # 输出 PPO 要追赶的上下界。

班次  风险顺序      全机器人动作  回报  最优动作      回报
 1    [1, 0, 1, 0, 1]    [0, 0, 0, 0, 0]  -7.0  [1, 0, 1, 0, 0]   5.0
 2    [0, 1, 0, 0, 1]    [0, 0, 0, 0, 0]  -3.0  [0, 1, 0, 0, 1]   9.0
 3    [1, 0, 1, 0, 0, 0]    [0, 0, 0, 0, 0, 0]  -2.0  [1, 0, 1, 0, 0, 0]  10.0
 4    [0, 1, 1, 0, 1, 0]    [0, 0, 0, 0, 0, 0]  -6.0  [0, 1, 1, 0, 0, 0]   6.0
 5    [1, 0, 0, 0]    [0, 0, 0, 0]   0.0  [1, 0, 0, 0]   6.0
 6    [0, 0, 1, 0, 0]    [0, 0, 0, 0, 0]   1.0  [0, 0, 1, 0, 0]   7.0
平均班次回报：全机器人=-2.8，可解释上界=7.2


## 核心实现一：手写 Actor-Critic、动作 mask 与轨迹采集

策略头输出“机器人/人工”两个 logits，价值头估计当前状态到班次末尾的期望回报。人工预算为零时，把人工 logit 置为极小值；采样用 `torch.multinomial`，并在采样当下保存 `old_log_prob`，后续 PPO epoch 不得改写它。

In [3]:
class ManualActorCritic(nn.Module):  # 定义共享隐藏层的手写策略价值网络。
    def __init__(self, state_dim=3, hidden_dim=16, action_count=2):  # 初始化共享层、actor 头与 critic 头。
        super().__init__()  # 注册基础模块状态。
        self.hidden_weight = nn.Parameter(torch.randn(state_dim, hidden_dim) * 0.25)  # 创建状态到隐藏层权重。
        self.hidden_bias = nn.Parameter(torch.zeros(hidden_dim))  # 创建共享隐藏偏置。
        self.actor_weight = nn.Parameter(torch.randn(hidden_dim, action_count) * 0.18)  # 创建动作 logits 权重。
        self.actor_bias = nn.Parameter(torch.zeros(action_count))  # 创建动作偏置。
        self.critic_weight = nn.Parameter(torch.randn(hidden_dim, 1) * 0.18)  # 创建状态价值权重。
        self.critic_bias = nn.Parameter(torch.zeros(1))  # 创建价值偏置。
    def forward(self, states, action_masks):  # 同时计算带合法性 mask 的策略 logits 与价值。
        hidden = torch.tanh(states @ self.hidden_weight + self.hidden_bias)  # 把三维业务状态映射到共享表示。
        logits = hidden @ self.actor_weight + self.actor_bias  # 计算机器人和人工两个动作分数。
        masked_logits = logits.masked_fill(~action_masks, -1e4)  # 在 softmax 前屏蔽预算不足的人工动作。
        values = (hidden @ self.critic_weight + self.critic_bias).squeeze(-1)  # 估计每个状态的标量价值。
        return masked_logits, values  # 返回策略与 critic 两个输出。
torch.manual_seed(79)  # 固定 Actor-Critic 初始参数。
agent = ManualActorCritic()  # 创建手写策略价值网络。
def collect_episode(model, sequence, stochastic=True):  # 用当前策略采集一个完整班次轨迹。
    states = []  # 收集采样时状态。
    masks = []  # 收集采样时合法动作 mask。
    actions = []  # 收集实际路由动作。
    rewards = []  # 收集即时业务收益。
    old_log_probs = []  # 冻结采样策略对动作的 log 概率。
    old_values = []  # 收集采样时 critic 估值。
    remaining_budget = 2  # 初始化两个人工名额。
    for step, risk in enumerate(sequence):  # 按时序处理当前班次全部工单。
        state = state_vector(risk, remaining_budget, len(sequence) - step, len(sequence))  # 构造当前决策状态。
        mask = torch.tensor([True, remaining_budget > 0])  # 预算耗尽时只允许机器人动作。
        logits, value = model(state.unsqueeze(0), mask.unsqueeze(0))  # 对单状态执行策略价值前向。
        probabilities = torch.softmax(logits[0], dim=-1)  # 得到合法动作概率。
        if stochastic:  # 训练采集需要按概率探索。
            action = int(torch.multinomial(probabilities, 1).item())  # 从当前策略分布抽样动作。
        else:  # 评估阶段使用确定性贪心动作。
            action = int(probabilities.argmax().item())  # 选择最大概率动作。
        reward = transition_reward(risk, action)  # 计算实际业务收益。
        states.append(state)  # 保存当前状态张量。
        masks.append(mask)  # 保存当前动作合法性合同。
        actions.append(action)  # 保存执行动作。
        rewards.append(reward)  # 保存即时收益。
        old_log_probs.append(torch.log(probabilities[action].clamp_min(1e-9)).detach())  # 冻结旧策略选中动作 log 概率。
        old_values.append(value[0].detach())  # 冻结采样时价值估计。
        remaining_budget -= int(action == 1)  # 人工动作后扣减预算。
    return states, masks, actions, rewards, old_log_probs, old_values  # 返回可用于 PPO 的完整轨迹字段。
initial_trajectory = collect_episode(agent, train_sequences[0])  # 用随机初始策略采集一条真实轨迹。
print("步  风险  剩余预算  动作    reward  old_logp  value")  # 输出采样账本表头。
for step, (state, action, reward, log_prob, value) in enumerate(zip(initial_trajectory[0], initial_trajectory[2], initial_trajectory[3], initial_trajectory[4], initial_trajectory[5])):  # 对齐首条轨迹各字段。
    print(f"{step:>2}   {int(state[0].item())}      {state[1].item() * 2:.0f}      {'人工' if action else '机器人':<3}   {reward:>5.1f}    {float(log_prob):>7.3f}  {float(value):>6.3f}")  # 展示策略采集时真正保存的数据。

步  风险  剩余预算  动作    reward  old_logp  value
 0   1      2      人工      3.0     -0.522  -0.022
 1   0      1      人工     -1.0     -0.610   0.026
 2   1      0      机器人    -3.0      0.000  -0.012
 3   0      0      机器人     1.0      0.000   0.012


## 核心实现二：GAE 与 PPO clipped objective

GAE 从后往前计算 $\delta_t=r_t+\gamma V_{t+1}-V_t$ 与 $A_t=\delta_t+\gamma\lambda A_{t+1}$。每个班次末尾的 `next_value=0`，因此不会跨 episode bootstrap。训练时把所有班次轨迹拼接，标准化 advantage，然后对同一批旧数据做数个 PPO epoch。

In [4]:
def compute_gae(rewards, values, gamma=0.95, gae_lambda=0.9):  # 为一个终止班次手写广义优势估计。
    advantages = torch.zeros(len(rewards))  # 创建与轨迹等长的优势张量。
    next_value = torch.tensor(0.0)  # episode 已终止所以末尾不得 bootstrap。
    running_advantage = torch.tensor(0.0)  # 初始化反向递推的累计优势。
    for index in range(len(rewards) - 1, -1, -1):  # 从最后一个决策反向遍历。
        delta = rewards[index] + gamma * next_value - values[index]  # 计算一步 TD 残差。
        running_advantage = delta + gamma * gae_lambda * running_advantage  # 累积带衰减的未来 TD 信息。
        advantages[index] = running_advantage  # 保存当前时刻优势。
        next_value = values[index]  # 前移 bootstrap 价值到上一时刻。
    returns = advantages + torch.stack(values)  # 用 A+V 得到 critic 监督目标。
    return advantages, returns  # 返回 actor 优势和 critic 回报。
initial_rewards = torch.tensor(initial_trajectory[3])  # 把首条轨迹奖励转换为张量。
initial_advantages, initial_returns = compute_gae(initial_rewards, initial_trajectory[5])  # 计算首条轨迹 GAE 与回报。
print("步  reward  value    advantage  return")  # 输出 GAE 中间账本表头。
for index, (reward, value, advantage, target_return) in enumerate(zip(initial_rewards, initial_trajectory[5], initial_advantages, initial_returns)):  # 对齐时序 TD 字段。
    print(f"{index:>2}  {float(reward):>6.2f}  {float(value):>7.3f}  {float(advantage):>9.3f}  {float(target_return):>7.3f}")  # 展示终止回传如何形成优势。
def gather_training_batch(model, sequences):  # 用当前旧策略收集一批完整 on-policy 班次。
    batch_states = []  # 收集所有状态。
    batch_masks = []  # 收集所有动作合法性 mask。
    batch_actions = []  # 收集所有采样动作。
    batch_old_log_probs = []  # 收集冻结旧 log 概率。
    batch_advantages = []  # 收集逐班次 GAE。
    batch_returns = []  # 收集逐班次 critic 目标。
    episode_returns = []  # 收集当前策略的班次业务回报。
    for sequence in sequences:  # 独立采集每个班次以正确处理 done。
        trajectory = collect_episode(model, sequence, stochastic=True)  # 获得当前旧策略轨迹。
        rewards = torch.tensor(trajectory[3])  # 转换即时奖励为张量。
        advantages, returns = compute_gae(rewards, trajectory[5])  # 在当前班次内部计算 GAE。
        batch_states.extend(trajectory[0])  # 追加状态。
        batch_masks.extend(trajectory[1])  # 追加动作 mask。
        batch_actions.extend(trajectory[2])  # 追加动作编号。
        batch_old_log_probs.extend(trajectory[4])  # 追加已冻结旧概率。
        batch_advantages.extend(advantages)  # 追加优势。
        batch_returns.extend(returns)  # 追加回报目标。
        episode_returns.append(float(rewards.sum()))  # 保存该班次采样收益。
    states = torch.stack(batch_states)  # 堆叠批量状态。
    masks = torch.stack(batch_masks)  # 堆叠批量合法动作 mask。
    actions = torch.tensor(batch_actions)  # 创建动作编号张量。
    old_log_probs = torch.stack(batch_old_log_probs).detach()  # 冻结旧策略概率避免反传。
    advantages = torch.stack(batch_advantages).detach()  # 冻结 GAE 避免 actor 改写目标。
    returns = torch.stack(batch_returns).detach()  # 冻结 critic 监督目标。
    advantages = (advantages - advantages.mean()) / advantages.std().clamp_min(1e-6)  # 标准化优势以稳定小批训练。
    return states, masks, actions, old_log_probs, advantages, returns, episode_returns  # 返回 PPO 更新所需全部字段。
print("初始 GAE 平均值 / 标准差：", round(float(initial_advantages.mean()), 3), round(float(initial_advantages.std()), 3))  # 展示优势不是硬编码标签。

步  reward  value    advantage  return
 0    3.00   -0.022      0.601    0.579
 1   -1.00    0.026     -2.860   -2.834
 2   -3.00   -0.012     -2.132   -2.144
 3    1.00    0.012      0.988    1.000
初始 GAE 平均值 / 标准差： -0.851 1.93


## 真实 PPO 训练与策略评估

每次 update 先重新采样六个班次，再冻结旧概率做 5 个 PPO epoch。下面打印策略比率、clip 比例、actor/critic loss 与采样回报轨迹，最后在六个测试顺序上贪心执行。

In [5]:
optimizer = torch.optim.Adam(agent.parameters(), lr=0.012)  # 创建同时更新 actor 与 critic 的 Adam。
training_history = []  # 保存每轮采样回报和 PPO 损失诊断。
clip_epsilon = 0.2  # 设置 PPO 概率比裁剪半径。
for update in range(120):  # 重复执行 on-policy 采样与多 epoch 更新。
    states, masks, actions, old_log_probs, advantages, returns, sampled_returns = gather_training_batch(agent, train_sequences)  # 用当前旧策略采集一批轨迹。
    last_actor_loss = torch.tensor(0.0)  # 初始化当前 update 的 actor loss 记录。
    last_critic_loss = torch.tensor(0.0)  # 初始化当前 update 的 critic loss 记录。
    last_ratio = torch.ones_like(advantages)  # 初始化策略概率比诊断。
    for _ in range(5):  # 在同一批冻结旧数据上做有限次 PPO 更新。
        optimizer.zero_grad()  # 清除上一 epoch 梯度。
        logits, values = agent(states, masks)  # 用新策略重新计算动作 logits 与价值。
        log_probabilities = torch.log_softmax(logits, dim=-1)  # 计算所有合法动作的 log 概率。
        new_log_probs = log_probabilities.gather(1, actions.unsqueeze(1)).squeeze(1)  # 取出采样动作在新策略下的 log 概率。
        ratios = torch.exp(new_log_probs - old_log_probs)  # 计算新旧策略动作概率比。
        unclipped_objective = ratios * advantages  # 计算未裁剪 surrogate objective。
        clipped_ratios = ratios.clamp(1.0 - clip_epsilon, 1.0 + clip_epsilon)  # 将过大策略变化裁剪到安全区间。
        clipped_objective = clipped_ratios * advantages  # 计算裁剪后的 surrogate objective。
        actor_loss = -torch.minimum(unclipped_objective, clipped_objective).mean()  # 最大化保守的两者较小目标。
        critic_loss = ((values - returns) ** 2).mean()  # 让 critic 拟合 GAE 形成的回报目标。
        probabilities = torch.softmax(logits, dim=-1)  # 计算策略概率以评估探索熵。
        entropy = -(probabilities * log_probabilities).sum(dim=-1).mean()  # 计算批量平均策略熵。
        total_loss = actor_loss + 0.45 * critic_loss - 0.015 * entropy  # 合并 actor、critic 和探索正则。
        total_loss.backward()  # 将 PPO 目标反向传播到共享网络。
        torch.nn.utils.clip_grad_norm_(agent.parameters(), 1.0)  # 限制极端小批梯度避免数值爆炸。
        optimizer.step()  # 更新手写 Actor-Critic 参数。
        last_actor_loss = actor_loss.detach()  # 保存本 update 最后 actor loss。
        last_critic_loss = critic_loss.detach()  # 保存本 update 最后 critic loss。
        last_ratio = ratios.detach()  # 保存真实新旧概率比。
    clipped_fraction = float(((last_ratio < 0.8) | (last_ratio > 1.2)).float().mean())  # 统计落在裁剪区外的样本比例。
    training_history.append((sum(sampled_returns) / len(sampled_returns), float(last_actor_loss), float(last_critic_loss), float(last_ratio.mean()), clipped_fraction))  # 保存本轮完整训练诊断。
ppo_returns = []  # 收集最终贪心策略测试回报。
ppo_actions = []  # 收集最终测试动作序列。
for sequence in test_sequences:  # 遍历六个测试班次。
    trajectory = collect_episode(agent, sequence, stochastic=False)  # 用确定性策略执行完整班次。
    ppo_actions.append(trajectory[2])  # 保存路由动作。
    ppo_returns.append(sum(trajectory[3]))  # 保存累计业务回报。
ppo_mean_return = sum(ppo_returns) / len(ppo_returns)  # 计算最终平均测试回报。
print("update  采样均值回报  actor_loss  critic_loss  ratio均值  越界比例")  # 输出 PPO 训练诊断表头。
for update in (0, 19, 59, 119):  # 选取四个关键训练阶段。
    row = training_history[update]  # 读取当前阶段诊断记录。
    print(f"{update + 1:>3}       {row[0]:>6.2f}       {row[1]:>8.4f}   {row[2]:>9.4f}    {row[3]:>6.3f}    {row[4]:>6.1%}")  # 展示回报和概率更新幅度。
print("班次  风险顺序      PPO动作       PPO回报  基线回报")  # 输出逐班次最终结果表头。
for index, (sequence, actions, ppo_return, baseline_return) in enumerate(zip(test_sequences, ppo_actions, ppo_returns, baseline_returns), start=1):  # 对齐策略结果和基线。
    readable_actions = ["人" if action else "机" for action in actions]  # 把动作编号转换为人工/机器人缩写。
    print(f"{index:>2}    {sequence}    {readable_actions}      {ppo_return:>5.1f}      {baseline_return:>5.1f}")  # 展示预算内的具体决策。
print(f"平均测试回报：全机器人={baseline_mean_return:.1f}，PPO={ppo_mean_return:.1f}，最优上界={oracle_mean_return:.1f}")  # 汇总策略学习收益。

update  采样均值回报  actor_loss  critic_loss  ratio均值  越界比例
  1         1.33        -0.0424     12.6243     1.021      4.2%
 20         6.67        -0.0214      3.7582     0.974      4.2%
 60         8.00        -0.0002      0.0408     1.000      0.0%
120         8.00         0.0000      0.0139     1.000      0.0%
班次  风险顺序      PPO动作       PPO回报  基线回报
 1    [1, 0, 1, 0, 1]    ['人', '机', '人', '机', '机']        5.0       -7.0
 2    [0, 1, 0, 0, 1]    ['机', '人', '机', '机', '人']        9.0       -3.0
 3    [1, 0, 1, 0, 0, 0]    ['人', '机', '人', '机', '机', '机']       10.0       -2.0
 4    [0, 1, 1, 0, 1, 0]    ['机', '人', '人', '机', '机', '机']        6.0       -6.0
 5    [1, 0, 0, 0]    ['人', '机', '机', '机']        6.0        0.0
 6    [0, 0, 1, 0, 0]    ['机', '机', '人', '机', '机']        7.0        1.0
平均测试回报：全机器人=-2.8，PPO=7.2，最优上界=7.2


## 结果解读

训练初期策略会随机浪费人工名额，随着 actor 和 critic 联合更新，采样回报向 8 靠近；最终动作表在未见 horizon 与风险比例上检查人工是否分给高风险工单以及预算是否超用。`ratio` 不应永远等于 1，`ratio 越界率` 说明同批多 epoch 更新走出了裁剪区间；是否真正选择 clipped surrogate 还必须结合 advantage 符号。两条测试序列含三张高风险而预算只有二，动作表会显示第三张被硬门禁回退；但所有高风险收益相同，实验仍不证明策略会根据未来严重度做前瞻取舍。教学环境只用于检查公式和账本，不宣称 PPO 一定优于规则。

## 失败案例：每个 epoch 重算“old log-prob”会让 ratio 永远为 1

旧策略概率必须来自采样时快照。若错误地用当前新策略同时充当 old 与 new，指数差恒为零，clip 机制失效。下面用一个独立、尚未饱和的真实 Actor-Critic 快照构造一次明显参数变化，对比冻结快照与错误重算。

In [6]:
torch.manual_seed(137)  # 固定独立策略探针的初始化与动作采样。
probe_agent = ManualActorCritic()  # 创建尚未饱和的独立 Actor-Critic 以观察真实裁剪。
probe_states, probe_masks, probe_actions, frozen_old_log_probs, probe_advantages, _, _ = gather_training_batch(probe_agent, train_sequences[:2])  # 用探针旧策略采集两班次并冻结概率。
with torch.no_grad():  # 人为制造一次明显策略更新且不记录梯度。
    probe_agent.actor_bias[1] += 2.5  # 大幅提高人工动作偏置以让部分概率比越过裁剪边界。
new_probe_logits, _ = probe_agent(probe_states, probe_masks)  # 用变化后的探针新策略重新计算 logits。
new_probe_log_probs = torch.log_softmax(new_probe_logits, dim=-1).gather(1, probe_actions.unsqueeze(1)).squeeze(1)  # 读取同一批采样动作的新概率。
correct_ratios = torch.exp(new_probe_log_probs - frozen_old_log_probs)  # 正确使用采样快照计算非一概率比。
wrong_old_log_probs = new_probe_log_probs.detach()  # 错误地把当前新策略概率冒充旧策略快照。
wrong_ratios = torch.exp(new_probe_log_probs - wrong_old_log_probs)  # 错误概率比恒为一。
correct_clipped = correct_ratios.clamp(0.8, 1.2)  # 对真实策略变化应用 PPO 裁剪。
wrong_clipped = wrong_ratios.clamp(0.8, 1.2)  # 对恒一错误概率比裁剪但不会产生作用。
ratio_outside_fraction = float(((correct_ratios < 0.8) | (correct_ratios > 1.2)).float().mean())  # 统计概率比落在裁剪区间外的比例。
positive_clipped = (probe_advantages >= 0.0) & (correct_ratios > 1.2)  # 正优势只有上穿边界时 clipped 项更保守。
negative_clipped = (probe_advantages < 0.0) & (correct_ratios < 0.8)  # 负优势只有下穿边界时 clipped 项更保守。
active_clip_fraction = float((positive_clipped | negative_clipped).float().mean())  # 统计真正由 min 选择 clipped surrogate 的比例。
print("样本  advantage  正确ratio  裁剪后  错误ratio")  # 输出快照错误对比表头。
for index in range(min(8, len(correct_ratios))):  # 展示前八个真实决策点。
    print(f"{index:>2}    {float(probe_advantages[index]):>7.3f}    {float(correct_ratios[index]):>7.3f}   {float(correct_clipped[index]):>6.3f}    {float(wrong_ratios[index]):>7.3f}")  # 展示新旧比率与裁剪差异。
print(f"正确方案非1比例={float((correct_ratios - 1.0).abs().gt(1e-4).float().mean()):.1%}，ratio越界={ratio_outside_fraction:.1%}，目标真正裁剪={active_clip_fraction:.1%}，错误方案非1比例={float((wrong_ratios - 1.0).abs().gt(1e-4).float().mean()):.1%}")  # 区分 ratio 越界、advantage 符号和真正生效的 clipped surrogate。

样本  advantage  正确ratio  裁剪后  错误ratio
 0      1.639      1.775    1.200      1.000
 1      0.716      0.144    0.800      1.000
 2      0.576      1.836    1.200      1.000
 3     -0.526      1.000    1.000      1.000
 4     -1.005      0.141    0.800      1.000
 5     -1.433      0.147    0.800      1.000
 6     -0.238      1.752    1.200      1.000
 7      0.270      1.857    1.200      1.000
正确方案非1比例=87.5%，ratio越界=87.5%，目标真正裁剪=62.5%，错误方案非1比例=0.0%


## 生产差距与追问

真实强化学习还需要离线安全评估、动作审批、reward hacking 监控、行为策略覆盖、截断与终止区分、并行环境、mini-batch shuffle、价值裁剪、KL early stop 和 checkpoint 回滚。客服路由通常先用可解释规则或受约束优化，PPO 只有在奖励可信、探索安全且可回放审计时才合适；人工名额是硬约束，必须由环境门禁而不是仅靠奖励学习。

## 最小回归测试

In [7]:
assert len(train_sequences) >= 5  # 保证强化学习案例覆盖多种到达顺序。
assert ppo_mean_return > baseline_mean_return  # 保护 PPO 策略确实改善全机器人基线。
assert all(sum(actions) <= 2 for actions in ppo_actions)  # 保护环境硬门禁没有超用人工预算。
assert active_clip_fraction > 0.0  # 保护结合 advantage 符号后确有样本使用 clipped surrogate。
assert float((wrong_ratios - 1.0).abs().max()) == 0.0  # 保护失败案例确实展示恒一概率比。
print("最小回归测试通过：GAE、预算门禁、PPO 收益和旧策略快照故障均保持有效。")  # 输出集中测试结论。

最小回归测试通过：GAE、预算门禁、PPO 收益和旧策略快照故障均保持有效。
